In [1]:
!git clone https://github.com/EleutherAI/lm-evaluation-harness
%cd lm-evaluation-harness
!pip install -e .
!pip install accelerate bitsandbytes

Cloning into 'lm-evaluation-harness'...
remote: Enumerating objects: 65297, done.
remote: Total 65297 (delta 0), reused 0 (delta 0), pack-reused 65297 (from 1)
Receiving objects: 100% (65297/65297), 35.14 MiB | 20.05 MiB/s, done.
Resolving deltas: 100% (45136/45136), done.
/content/lm-evaluation-harness
Obtaining file:///content/lm-evaluation-harness
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 16.2 MB/s eta 0:00:

In [2]:
import json, os, glob
from huggingface_hub import snapshot_download
from safetensors.torch import load_file, save_file
from transformers import AutoConfig, AutoTokenizer

REPO = "arefe-rn/Ministral-3-8B_SFT_16bit"
OUT = "/kaggle/temp/ministral_sft_text"
os.makedirs(OUT, exist_ok=True)

src = snapshot_download(REPO)
cfg = AutoConfig.from_pretrained(src)
print("model_type:", cfg.model_type, "| text:", cfg.text_config.model_type)
print(list(load_file(sorted(glob.glob(f"{src}/*.safetensors"))[0]).keys())[:8])

def remap(k):
    if "vision_tower" in k or "multi_modal_projector" in k:
        return None
    for pre in ("language_model.model.", "model.language_model."):
        if k.startswith(pre):
            return "model." + k[len(pre):]
    if k.startswith("language_model.lm_head."):
        return "lm_head." + k[len("language_model.lm_head."):]
    return k

weight_map = {}
for f in sorted(glob.glob(f"{src}/*.safetensors")):
    sd = load_file(f)
    new = {remap(k): v.contiguous() for k, v in sd.items() if remap(k) is not None}
    name = os.path.basename(f)
    save_file(new, f"{OUT}/{name}", metadata={"format": "pt"})
    weight_map.update({k: name for k in new})
    del sd, new

json.dump({"metadata": {}, "weight_map": weight_map},
          open(f"{OUT}/model.safetensors.index.json", "w"))

tc = cfg.text_config.to_dict()
tc["architectures"] = ["Ministral3ForCausalLM"]
tc["torch_dtype"] = "bfloat16"
tc["sliding_window"] = tc.get("sliding_window") or tc.get("max_position_embeddings") or 262144
tc["layer_types"] = ["full_attention"] * tc["num_hidden_layers"]
json.dump(tc, open(f"{OUT}/config.json", "w"), indent=2)

AutoTokenizer.from_pretrained(src, fix_mistral_regex=True).save_pretrained(OUT)
print("done:", len(weight_map), "tensors")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}


model_type: mistral3 | text: ministral
['language_model.model.embed_tokens.weight', 'language_model.model.layers.0.input_layernorm.weight', 'language_model.model.layers.0.mlp.down_proj.weight', 'language_model.model.layers.0.mlp.gate_proj.weight', 'language_model.model.layers.0.mlp.up_proj.weight', 'language_model.model.layers.0.post_attention_layernorm.weight', 'language_model.model.layers.0.self_attn.k_proj.weight', 'language_model.model.layers.0.self_attn.o_proj.weight']


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}


done: 309 tensors


## English GSM8K tasks

In [ ]:
!lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/temp/ministral_sft_text,parallelize=True,dtype=bfloat16 \
    --tasks gsm8k gsm8k_cot gsm8k_cot_zeroshot gsm8k_cot_llama\
    --batch_size 1 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/results

2026-10-19:18:14:15 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-19:18:14:28 INFO     [_cli.run:413] Selected Tasks: ['gsm8k', 'gsm8k_cot', 'gsm8k_cot_zeroshot', 'gsm8k_cot_llama']
2026-10-19:18:14:31 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-19:18:14:31 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': '/kaggle/temp/ministral_sft_text', 'parallelize': True, 'dtype': 'bfloat16'}
2026-10-19:18:14:38 INFO     [models.huggingface:309] Using `accelerate launch` or `parallelize=True`, device 'cuda:0' will be overridden when placing model.
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
2026-10-19:18:1

## Persian gsm8k tasks

In [ ]:

!lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/temp/ministral_sft_text,parallelize=True,dtype=bfloat16 \
    --tasks persian_gsm8k persian_gsm8k_cot persian_gsm8k_cot_zeroshot \
    --include_path /kaggle/input/datasets/amirint/pgsm8k \
    --batch_size 1 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/results_persian

2026-10-20:06:05:30 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-20:06:05:44 WARNING  [tasks._index:70] Task 'chain_of_thought' from None overrides existing task from None
2026-10-20:06:05:44 WARNING  [tasks._index:70] Task 'math_word_problems' from None overrides existing task from None
2026-10-20:06:05:44 INFO     [_cli.run:412] Including path: /kaggle/input/datasets/amirint/pgsm8k
2026-10-20:06:05:44 INFO     [_cli.run:413] Selected Tasks: ['persian_gsm8k', 'persian_gsm8k_cot', 'persian_gsm8k_cot_zeroshot']
2026-10-20:06:05:49 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-20:06:05:49 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': '/kaggle/temp/ministral_sft_text', 'parallelize': True, 'dtype': 'bfloat16'}
2026-10-20:06:05:57 INFO     [models.hugging

## hendrycks_math Tasks

In [3]:
%%writefile /content/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py
from typing import Dict, List

import datasets


def process_docs(dataset: datasets.Dataset) -> datasets.Dataset:
    def _process_doc(doc: dict) -> dict:
        out_doc = {
            "problem": doc["problem"],
            "solution": doc["solution"],
            "answer": remove_boxed(last_boxed_only_string(doc["solution"])),
        }
        return out_doc

    return dataset.map(_process_doc)


def process_results(doc: dict, results: List[str]) -> Dict[str, int]:
    retval = 0
    response = results[0]

    # Try to extract \boxed{} from the response (matches aime task behavior)
    boxed = last_boxed_only_string(response)
    if boxed is not None:
        try:
            answer = remove_boxed(boxed)
        except AssertionError:
            answer = None
    else:
        answer = None

    # Fall back to $...$ extraction
    if answer is None:
        indices = [pos for pos, char in enumerate(response) if char == "$"]
        if len(indices) <= 1:
            answer = response
        else:
            answer = response[indices[0] + 1 : indices[-1]]

    if is_equiv(answer, remove_boxed(last_boxed_only_string(doc["solution"]))):
        retval = 1

    results = {
        "exact_match": retval,
    }
    return results


# string normalization from https://github.com/EleutherAI/lm-evaluation-harness/blob/master/lm_eval/tasks/hendrycks_math.py
def is_equiv(str1, str2, verbose=False):
    if str1 is None and str2 is None:
        print("WARNING: Both None")
        return True
    if str1 is None or str2 is None:
        return False

    try:
        ss1 = strip_string(str1)
        ss2 = strip_string(str2)
        if verbose:
            print(ss1, ss2)
        return ss1 == ss2
    except Exception:
        return str1 == str2


def remove_boxed(s):
    if "\\boxed " in s:
        left = "\\boxed "
        assert s[: len(left)] == left
        return s[len(left) :]

    left = "\\boxed{"

    assert s[: len(left)] == left
    assert s[-1] == "}"

    return s[len(left) : -1]


def last_boxed_only_string(string):
    idx = string.rfind("\\boxed")
    if "\\boxed " in string:
        return "\\boxed " + string.split("\\boxed ")[-1].split("$")[0]
    if idx < 0:
        idx = string.rfind("\\fbox")
        if idx < 0:
            return None

    i = idx
    right_brace_idx = None
    num_left_braces_open = 0
    while i < len(string):
        if string[i] == "{":
            num_left_braces_open += 1
        if string[i] == "}":
            num_left_braces_open -= 1
            if num_left_braces_open == 0:
                right_brace_idx = i
                break
        i += 1

    if right_brace_idx is None:
        retval = None
    else:
        retval = string[idx : right_brace_idx + 1]

    return retval


def fix_fracs(string):
    substrs = string.split("\\frac")
    new_str = substrs[0]
    if len(substrs) > 1:
        substrs = substrs[1:]
        for substr in substrs:
            new_str += "\\frac"
            if substr[0] == "{":
                new_str += substr
            else:
                try:
                    assert len(substr) >= 2
                except AssertionError:
                    return string
                a = substr[0]
                b = substr[1]
                if b != "{":
                    if len(substr) > 2:
                        post_substr = substr[2:]
                        new_str += "{" + a + "}{" + b + "}" + post_substr
                    else:
                        new_str += "{" + a + "}{" + b + "}"
                else:
                    if len(substr) > 2:
                        post_substr = substr[2:]
                        new_str += "{" + a + "}" + b + post_substr
                    else:
                        new_str += "{" + a + "}" + b
    string = new_str
    return string


def fix_a_slash_b(string):
    if len(string.split("/")) != 2:
        return string
    a = string.split("/")[0]
    b = string.split("/")[1]
    try:
        a = int(a)
        b = int(b)
        assert string == "{}/{}".format(a, b)
        new_string = "\\frac{" + str(a) + "}{" + str(b) + "}"
        return new_string
    except AssertionError:
        return string


def remove_right_units(string):
    # "\\text{ " only ever occurs (at least in the val set) when describing units
    if "\\text{ " in string:
        splits = string.split("\\text{ ")
        assert len(splits) == 2
        return splits[0]
    else:
        return string


def fix_sqrt(string):
    if "\\sqrt" not in string:
        return string
    splits = string.split("\\sqrt")
    new_string = splits[0]
    for split in splits[1:]:
        if split[0] != "{":
            a = split[0]
            new_substr = "\\sqrt{" + a + "}" + split[1:]
        else:
            new_substr = "\\sqrt" + split
        new_string += new_substr
    return new_string


def strip_string(string):
    # linebreaks
    string = string.replace("\n", "")

    # remove inverse spaces
    string = string.replace("\\!", "")

    # replace \\ with \
    string = string.replace("\\\\", "\\")

    # replace tfrac and dfrac with frac
    string = string.replace("tfrac", "frac")
    string = string.replace("dfrac", "frac")

    # remove \left and \right
    string = string.replace("\\left", "")
    string = string.replace("\\right", "")

    # Remove circ (degrees)
    string = string.replace("^{\\circ}", "")
    string = string.replace("^\\circ", "")

    # remove dollar signs
    string = string.replace("\\$", "")

    # remove units (on the right)
    string = remove_right_units(string)

    # remove percentage
    string = string.replace("\\%", "")
    string = string.replace("\%", "")  # noqa: W605

    # " 0." equivalent to " ." and "{0." equivalent to "{." Alternatively, add "0" if "." is the start of the string
    string = string.replace(" .", " 0.")
    string = string.replace("{.", "{0.")
    # if empty, return empty string
    if len(string) == 0:
        return string
    if string[0] == ".":
        string = "0" + string

    # to consider: get rid of e.g. "k = " or "q = " at beginning
    if len(string.split("=")) == 2:
        if len(string.split("=")[0]) <= 2:
            string = string.split("=")[1]

    # fix sqrt3 --> sqrt{3}
    string = fix_sqrt(string)

    # remove spaces
    string = string.replace(" ", "")

    # \frac1b or \frac12 --> \frac{1}{b} and \frac{1}{2}, etc. Even works with \frac1{72} (but not \frac{72}1). Also does a/b --> \\frac{a}{b}
    string = fix_fracs(string)

    # manually change 0.5 --> \frac{1}{2}
    if string == "0.5":
        string = "\\frac{1}{2}"

    # NOTE: X/Y changed to \frac{X}{Y} in dataset, but in simple cases fix in case the model output is X/Y
    string = fix_a_slash_b(string)

    return string

Overwriting /content/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py


In [ ]:
!accelerate launch --num_processes=1 -m lm_eval --model hf \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --model_args pretrained=/kaggle/temp/ministral_sft_text,parallelize=True,dtype=bfloat16 \
    --gen_kwargs max_new_tokens=1024 \
    --limit 5 \
    --log_samples \
    --batch_size 2 \
    --device cuda:0 \
    --output_path /kaggle/working/results/hendricks \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Use standard mathematical notation for fractions, radicals, equations, and other expressions. Put your final answer in \boxed{...}."

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-20:09:11:36 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-20:09:11:49 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-20:09:11:50 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-20:09:11:50 WARNING  [evaluator:226] generation_kwargs: {'ma

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True accelerate launch --num_processes=1 -m lm_eval --model hf \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --model_args pretrained=/kaggle/temp/ministral_sft_text,parallelize=True,dtype=float16 \
    --gen_kwargs max_new_tokens=1024 \
    --limit 100 \
    --log_samples \
    --num_fewshot 1 \
    --batch_size 1 \
    --output_path /kaggle/working/results/hendricks \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Use standard mathematical notation for fractions, radicals, equations, and other expressions. Put your final answer in \boxed{...}."

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-20:11:39:25 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-20:11:39:35 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-20:11:39:37 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-20:11:39:37 WARNING  [evaluator:226] generation_kwargs: {'ma

In [5]:
!python -m lm_eval --model hf \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --model_args pretrained=/kaggle/temp/ministral_sft_text,dtype=bfloat16 \
    --device cuda:0 \
    --gen_kwargs max_new_tokens=1024 \
    --limit 100 \
    --log_samples \
    --num_fewshot 1 \
    --batch_size 2 \
    --output_path /content/hendricks \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Use standard mathematical notation for fractions, radicals, equations, and other expressions. Put your final answer in \boxed{...}."

2026-10-20:17:16:04 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-20:17:16:14 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-20:17:16:15 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-20:17:16:15 WARNING  [evaluator:226] generation_kwargs: {'max_new_tokens': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-10-20:17:16:15 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': '/kaggle/temp/ministral_sft_text', 'dtype': 'bfloat16'}
2026-10-20:17:16:20 INFO     [mod

In [4]:
import torch
print(torch.__version__)           # a "+cpu" suffix means CPU-only build
print(torch.version.cuda)          # None = no CUDA support
print(torch.cuda.is_available())

2.11.0+cu130
13.0
True


In [6]:
!nvidia-smi

Tue Oct 20 18:08:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P0             56W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [5]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True accelerate launch --num_processes=1 -m lm_eval --model hf \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --model_args pretrained=/kaggle/temp/ministral_sft_text,parallelize=True,dtype=bfloat16 \
    --gen_kwargs max_new_tokens=1024 \
    --limit 100 \
    --log_samples \
    --num_fewshot 1 \
    --batch_size 2 \
    --output_path /content/hendricks \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Use standard mathematical notation for fractions, radicals, equations, and other expressions. Put your final answer in \boxed{...}."

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-20:16:58:29 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-20:16:58:42 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-20:16:58:45 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-20:16:58:45 WARNING  [evaluator:226] generation_kwargs: {'ma

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True accelerate launch --num_processes=1 -m lm_eval --model hf \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --model_args pretrained=/kaggle/temp/ministral_sft_text,parallelize=True,dtype=float16 \
    --gen_kwargs max_new_tokens=1024 \
    --limit 100 \
    --log_samples \
    --num_fewshot 1 \
    --batch_size 2 \
    --apply_chat_template \
    --output_path /kaggle/working/results/hendricks \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Use standard mathematical notation for fractions, radicals, equations, and other expressions. Put your final answer in \boxed{...}."

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-20:12:40:04 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-20:12:40:04 INFO     [config.evaluate_config:364] Using default fewshot_as_multiturn=True.
2026-10-20:12:40:14 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-20:12:40:16 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setti